In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
Q1_path = os.path.join(path + "/Q1_data.csv")
df = pd.read_csv(Q1_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=20, edgecolor='purple')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df = df.drop("Order_ID",axis=1)

In [ ]:
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0])
if missing_values.any():
    print("\nHandle Missing Values as needed.")
else:
    print("\nNo Missing Values Found.")

In [ ]:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
  print("Dropping Duplicates...")
  df.drop_duplicates(inplace=True)
  print("Duplicates Dropped.")
else:
  print("No Duplicate Samples Found.")

In [ ]:
label_encoders = {}
for col in df:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

In [ ]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
print("Target Distribution:")
print(df["Delivery_Time"].value_counts(normalize=True)) #value_counts() → counts how many times each unique value appears,
# normalize=True → converts counts to fractions of the total (i.e., probabilities)
sns.countplot(x=df["Delivery_Time"])
plt.title("Target Distribution")
plt.show()


In [ ]:
X = df.drop("Delivery_Time", axis=1).astype(float) # converting all columns to float
y = df['Delivery_Time'].astype(float)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error

# Define K-Fold Cross Validation
n_splits = 5
kf = KFold(n_splits, shuffle=True, random_state=42)

avg_fold = 0.0
# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1): #?????
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)


    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    avg_fold = avg_fold + mae
    print("Avg MAE is ",mae)
    print("-" * 30)

avg_fold = avg_fold / n_splits
print("The AVG Of All MAE folds is ", avg_fold)





In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

sklearn_models = {
  "K-Nearest Neighbors": KNeighborsClassifier(
      n_neighbors=3,  # Number of neighbors to consider
  ),
  "Support Vector Machine": SVC(
      kernel='rbf',  # 'linear', 'poly', 'rbf', 'sigmoid'
      C=0.75  # Regularization parameter
  ),
  "Decision Tree": DecisionTreeClassifier(
      max_depth=3  # Maximum depth of tree (prevents overfitting)
  ),
  "Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  )
}

importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_
importances['KFold'] = sklearn_models['KFold'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: